In [ ]:
#| default_exp machine_learning.llm_notation_summarization

Previously, `trouver` used fine-tuned Transformers based models (Specifically Google's T5) to summarize notation. However, procuring high quality data is highly time consuming for this approach and the fine-tuned models have yet produce generally competent predictions.

This module instead opts to use large language models (LLM's)/generative AI models for the summarization task.

In [ ]:
#| export
from os import PathLike
from typing import Optional, TypedDict
import re

import lmstudio as lms
from lmstudio import LLM

from trouver.obsidian.file import MarkdownFile, MarkdownLineEnum
from trouver.llm_core.call_llm import call_llm, process_llm_response, smart_truncate, SupportedLLM, LLMResponse
from trouver.machine_learning.notation_summarization import _summary_should_be_generated, _get_summary, get_latex_in_original_from_parsed_notation_note_data, single_input_for_notation_summarization, notation_summarization_data_from_note, _write_summary_to_notation_note, format_training_tokens, format_for_gemma_instruct
from trouver.personal_vault.note_processing import process_standard_information_note
from trouver.notation.parse import main_of_notation, parse_notation_note
from trouver.obsidian.vault import VaultNote
from trouver.obsidian.links import links_from_text


In [ ]:
from trouver.helper.tests import _test_directory

In [ ]:
#| export

# --- 1. SYSTEM PROMPT (TARGET + TEXT SPLIT) ---
NOTATION_SUMMARIZATION_SYSTEM_PROMPT = r"""
# Role
You are a strict, non-conversational mathematical compiler. Your task is to extract a rigorous definition summary from the provided text for a specific **Target Symbol**.

# Input Data
1.  **Text:** The raw mathematical excerpt containing the definition.
2.  **Target:** The specific notation to extract and define from the text above.

# Goal
Generate the prose continuation of a sentence starting with: "$SYMBOL$ [[SOURCE|denotes]]..."
**Start your output immediately with the noun phrase.** The output must be mathematically precise, grammatically correct prose where all mathematics in valid MathJax/LaTeX that is renderable in Obsidian.md markdown. Your summary must prioritize information explicitly stated in the Text. While you may use internal knowledge to provide standard notation (like codomains) for clarity, you must not include auxiliary theorems or external properties that the Text does not discuss.

# Generation Rules

## 1. The "Identity Signature"
- **Priority:** Use the formal name if given (e.g., "the norm," "the sheaf of differentials").
- **Parent Context:** Immediately link the symbol to its primary parameters in the first sentence (e.g., "of a morphism $f$," "of an ideal $\mathfrak{a}$").
- **Minimalist Context**: If the Text defines the Target and then shifts to defining other auxiliary objects, do not summarize those auxiliary objects unless they are strictly necessary to understand the construction of the Target itself.

## 2. The Operational Definition
- Explain **how** it is constructed using the **exact abstraction level** of the text.
- **Justification:** Keep "because" clauses that explain well-definedness (e.g., "finite because it is full-rank").
- **Procedural Logic:** If the text lists steps (1, 2, 3), use a numbered list.
- **Formulas:** Include the defining equation.

## 3. Context Reconstruction ("Where" Clause) 
- **Recursive Unwinding:** Do not just list variables; **briefly define them**.   
- *Bad:* "...where $cl_X$ is the cycle class map."  
- *Good:* "...where $cl_X: CH^i(X) \to H^{2i}(X)$ is the cycle class map."
 - **Integration:** Use a "where..." clause immediately following the definition to define variables like $K, \mathcal{O}_K$, $X, Y$.

## 4. Structural Synthesis (Source-Tethered)
- **New Paragraph**: Use a new paragraph to separate the core construction from distinct properties, characterizations, or theorems elaborated on in the text.
- **Tethering**: Every point made in these paragraphs must have a direct "hook" in the provided Text. If the text defines a symbol but doesn't mention its properties, your output should be a single concise paragraph.

## 5. Constraints
- **Tone:** Clinical, precise. No "We defined," "Note that."
- **Safety:** ABSOLUTELY NO EMOJIS. NO MARKDOWN CODE BLOCKS.
- **Scope Control**: Avoid "encyclopedic overreach." Do not add long lists of properties (e.g., "The norm is multiplicative," "The topos is compact") unless the provided Text specifically treats those properties as part of its own narrative. Synthesis should clarify the source, not replace it.
- **Note Scope**: The note is about the Target, not necessarily the entire Text. Only talk about the content of the Text to the extent that it defines the Target and discusses its important properties. If the Target is simply a base object, identify it and stop. 

## 6. LaTeX Standardization (MANDATORY)
- **Error Correction:** You must fix syntactic errors in the input. If the text has `\frac a b` (ambiguous) or `\alph` (typo), output correct standard LaTeX: `\frac{a}{b}`, `\alpha`.
- **Obsidian Format:**
  - Use `$` for inline math.
  - Use `$$` for block math/equations.
  - Use `$$\begin{align*} ... \end{align*}$$` for multi-line definitions.
  - NEVER use `\(` `\)` `\[` `\]`.
- **Notation:**
  - Convert plain text operators to commands: "Gal" -> `\operatorname{Gal}`, "Hom" -> `\operatorname{Hom}` (or use \mathrm or \mathbf, etc. as appropriate for the text).

## 7. Input Sanitization

    Text Cleaning: If the input text contains typos (e.g., "morphism off schemes", "defind as"), correct the spelling in your prose output.

    Variable Consistency: If the input text uses inconsistent notation for the same object (e.g., switching between $\epsilon$ and $\varepsilon$ randomly), standardize to the most common or standard usage in the definition.

    Noise Removal: Ignore filler words like "clearly," "obviously," or conversational asides found in the source text.
---

# Example 1: Constructive Object
**Text:**
Let $f: X \to Y$ be a morphism of schemes... The sheaf of relative differentials, denoted $\Omega_{X/Y}$, is then defined to be the pullback of this conormal sheaf... That is, $\Omega_{X/Y} := \Delta^*(\mathcal{I}/\mathcal{I}^2)$.
**Target:** \Omega_{X/Y}
**Output:**
the sheaf of relative differentials of a morphism $f: X \to Y$ of schemes. It is defined as the pullback $\Omega_{X/Y} := \Delta^*(\mathcal{I}/\mathcal{I}^2)$, where $\mathcal{I}$ is the ideal sheaf of the diagonal immersion $\Delta: X \to X \times_Y X$. Since the conormal sheaf $\mathcal{I}/\mathcal{I}^2$ is a sheaf on the diagonal $\Delta(X)$ (viewed as a sheaf on $X$), the result is a sheaf on $X$.

An important property is that for affine open subschemes $U \subseteq X$ and $V \subseteq Y$ with $f(U) \subseteq V$, the sections $\Omega_{X/Y}(U)$ are isomorphic to the module of relative differentials $\Omega_{A/R}$.

# Example 2: Property/Value
**Text:**
Let $K$ be a number field... We define the norm of an ideal $\mathfrak{a}$, denoted $N(\mathfrak{a})$, to be the cardinality of the quotient ring $\mathcal{O}_K / \mathfrak{a}$. Since $\mathfrak{a}$ is a full-rank sublattice of $\mathcal{O}_K$, this quotient ring is always finite.
**Target:** N(\mathfrak{a})
**Output:**
the norm of a non-zero ideal $\mathfrak{a}$ in the ring of integers $\mathcal{O}_K$ of a number field $K$. It is defined as the cardinality of the quotient ring $N(\mathfrak{a}) := |\mathcal{O}_K / \mathfrak{a}|$. This quotient is always finite because $\mathfrak{a}$ is a full-rank sublattice of $\mathcal{O}_K$.

# Example 3: Procedural Definition
**Text:**
Thus if $D \in \operatorname{Div}(V)$ is a divisor and we want to choose a particular height function $h_{V, D}$, we need to make the following choices:
[1] Choose very ample divisors $D_{1}$ and $D_{2}$ with $D=D_{1}-D_{2}$.
...
**Target:** h_{V,D}
**Output:**
a particular height function for a divisor $D$ on a variety $V$. It is defined by the following construction:
1. Choose very ample divisors $D_{1}$ and $D_{2}$ such that $D=D_{1}-D_{2}$.
2. Choose embeddings $\phi_{1}: V \rightarrow \mathbb{P}^{n}$ and $\phi_{2}: V \rightarrow \mathbb{P}^{m}$ corresponding respectively to $D_{1}$ and $D_{2}$.
3. Set $h_{V, D}(P)=h\left(\phi_{1}(P)\right)-h\left(\phi_{2}(P)\right)$.

# Example 4: Inconsistency Standardization
**Text:**
The set of morphisms between schemes X and Y is usually writen Hom(X,Y). We say that a morphism $f \in \operatorname{Hom}(X, Y)$ is separated if the diagonal is a closed immersion.
**Target:** \operatorname{Hom}(X,Y)
**Output:**
the set of morphisms between schemes $X$ and $Y$. 

"""


In [ ]:
#| export
def example_from_notation_note(
        notation_note: VaultNote
        ) -> dict[str, str]: # Keys include 'processed_main_note_content' and 'latex_in_original'
    """
    Helper to parse `notation_note` and gather data for summarization.
    
    This function extracts the LaTeX used in the original main note and 
    retrieves the processed content of that main note.
    """
    parsed = parse_notation_note(notation_note)

    # Access the main note (the one where the notation is defined)
    main_note = VaultNote(
        vault=notation_note.vault, name=parsed.name_of_main_note)
    
    # Process the main note's content to remove metadata/formatting
    main_note_mf = MarkdownFile.from_vault_note(main_note)
    main_note_content = str(process_standard_information_note(main_note_mf, notation_note.vault))

    # Identify the specific LaTeX string as it appeared in the source
    latex_in_original = get_latex_in_original_from_parsed_notation_note_data(
            parsed.yaml_frontmatter_meta, parsed.notation_str)
    
    # Extract existing summary data
    summary_data = notation_summarization_data_from_note(
        notation_note, notation_note.vault, check_for_actual_summarization=False)
    
    return summary_data

In [ ]:
# Setup a path to your test vault
test_vault_path = _test_directory() / 'test_vault_4'
# Example notation note: 'number_theory_reference_1_notation_Z_nZ_ring_of_integers_modulo_n'
example_note = VaultNote(test_vault_path, name='number_theory_reference_1_notation_Z_nZ_ring_of_integers_modulo_n')

# Run the helper
data = example_from_notation_note(example_note)

# print(data.keys)
# print(f"Notation: {data['notation']}")
print(f"Main Note: {data['main_note_name']}")
print(f"Content: {data['processed_main_note_content']}")
print(f"latex_in_original: {data['latex_in_original']}")
# Output would show the extracted dictionary data

Main Note: number_theory_reference_1_Definition 1.7
Content: The ring of integers modulo $n$, denoted $\mathbb{Z}/n\mathbb{Z}$ has the elements ...

latex_in_original: $\mathbb{Z}/n\mathbb{Z}$


C:\Users\hyunj\Documents\Development\Python\trouver\trouver\helper\html.py:99: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  parsed_soup = BeautifulSoup(text, 'html.parser')


In [ ]:
#| hide
from fastcore.test import test_eq, test_is

# 1. Test basic functionality with a known note
test_vault = _test_directory() / 'test_vault_4'
note_name = r'number_theory_reference_1_notation_Z_nZ_ring_of_integers_modulo_n'
notat_note = VaultNote(test_vault, name=note_name)

res = example_from_notation_note(notat_note)

# Verify expected keys in return dict
expected_keys = {"notation_note_name", "notation", "latex_in_original", "summary", "main_note_name"}
assert expected_keys.issubset(res.keys())

# Verify specific values based on the vault content
test_eq(res['main_note_name'], 'number_theory_reference_1_Definition 1.7')
test_eq(res['notation'], r'$\mathbb{Z}/n\mathbb{Z}$')

# 2. Test behavior with different check_for_actual_summarization logic
# If the note isn't summarized yet, ensure the helper still returns data when False
res_no_check = example_from_notation_note(notat_note)
assert res_no_check is not None

In [ ]:
#| export

#| export
# def notation_note_should_be_summarized(
#         notation_note: VaultNote,
#         vault: PathLike,
#         main_note: Optional[VaultNote] = None, # The main note from which the notation comes from. If this is `None`, then the `main_note` is obtained via the `main_of_notation` function.
#         overwrite_previous_autogenerated_summary: bool = False, # If `True`, overwrite previously autogenerated summaries
#         latex_in_original_comes_first: bool = True # This is a parameter to pass to calls to the `single_input_for_notation_summarization` function. If `True`, the `latex_in_original` piece appears before the `main_note_content`. While the default value of `True` is recommended, passing `False` to this parameter may be necessary to use the older version of the summarization model in the repo [`notation_summarizations_model`](https://huggingface.co/hyunjongkimmath/notation_summarizations_model).
#     ) -> bool:
#     r"""
#     Return `True` if notation note has no summary or an autogenerated summary.

#     Helper to tell
#     """
#     metadata, notation_str, main_note_name,\
#         notation_note_content_mf, _\
#         = parse_notation_note(notation_note, vault)
#     summary_should_be_generated, main_mf = _summary_should_be_generated(
#         main_note, main_note_name, vault, notation_note,
#         overwrite_previous_autogenerated_summary,
#         metadata, notation_note_content_mf)
#     return summary_should_be_generated

def notation_note_should_be_summarized(
        notation_note: VaultNote,
        vault: PathLike,
        main_note: Optional[VaultNote] = None, # The main note from which the notation comes from. If this is `None`, then the `main_note` is obtained via the `main_of_notation` function.
        overwrite_previous_autogenerated_summary: bool = False, # If `True`, overwrite previously autogenerated summaries
        latex_in_original_comes_first: bool = True # This is a parameter to pass to calls to the `single_input_for_notation_summarization` function. If `True`, the `latex_in_original` piece appears before the `main_note_content`. While the default value of `True` is recommended, passing `False` to this parameter may be necessary to use the older version of the summarization model in the repo [`notation_summarizations_model`](https://huggingface.co/hyunjongkimmath/notation_summarizations_model).
    ) -> bool:
    r"""
    Return `True` if a notation note has no summary or contains an autogenerated summary that should be updated.

    This helper determines if the note is a candidate for the summarization pipeline based on:
    1. The presence of the `_auto/notation_summary` tag (if `overwrite_previous_autogenerated_summary` is True).
    2. Whether the note currently lacks content beyond the "denotes" link.
    3. Whether the main reference note actually exists and contains content.
    """
    # parse_notation_note retrieves metadata, the notation string, and the main note's name
    metadata, notation_str, main_note_name,\
        notation_note_content_mf, _\
        = parse_notation_note(notation_note, vault)
        
    # _summary_should_be_generated performs the logic of checking if the main note exists
    # and if the notation note is already "sufficiently summarized" or tagged as auto-generated.
    summary_should_be_generated, _ = _summary_should_be_generated(
        main_note, main_note_name, vault, notation_note,
        overwrite_previous_autogenerated_summary,
        metadata, notation_note_content_mf)
        
    return summary_should_be_generated

In [ ]:
# Setup test environment
test_vault = _test_directory() / 'test_vault_4'
# A note that has already been summarized manually
manual_note = VaultNote(test_vault, name='number_theory_reference_1_notation_Z_nZ_ring_of_integers_modulo_n')

# Check if it needs summarization (it shouldn't, as it has manual content)
needs_summary = notation_note_should_be_summarized(manual_note, test_vault)
print(f"Needs summary: {needs_summary}") 

# A note that only has the "denotes" link and no actual explanation
# empty_note = VaultNote(test_vault, name='some_new_unsummarized_notation')
# print(f"Empty note needs summary: {notation_note_should_be_summarized(empty_note, test_vault)}")

Needs summary: False


C:\Users\hyunj\Documents\Development\Python\trouver\trouver\machine_learning\notation_summarization.py:557: UserWarning: The notation note already has contents, so no new summary was added. Notation note name: number_theory_reference_1_notation_Z_nZ_ring_of_integers_modulo_n
  warnings.warn(


In [ ]:
#| hide
from fastcore.test import test_eq
import tempfile
import shutil
from pathlib import Path

# 1. Test with existing summarized note
test_vault = _test_directory() / 'test_vault_4'
summarized_note = VaultNote(test_vault, name='number_theory_reference_1_notation_Z_nZ_ring_of_integers_modulo_n')

# Should be False because it's already summarized and not tagged as _auto
test_eq(notation_note_should_be_summarized(summarized_note, test_vault), False)

# 2. Test with an auto-generated note
with tempfile.TemporaryDirectory() as temp_dir:
    temp_path = Path(temp_dir)
    shutil.copytree(test_vault, temp_path / 'vault')
    v = temp_path / 'vault'
    
    # Simulate an auto-generated note by adding the tag
    auto_note = VaultNote(v, name='number_theory_reference_1_notation_Z_nZ_ring_of_integers_modulo_n')
    mf = MarkdownFile.from_vault_note(auto_note)
    mf.add_tags(['_auto/notation_summary'])
    mf.write(auto_note)
    
    # By default, it shouldn't overwrite
    test_eq(notation_note_should_be_summarized(auto_note, v, overwrite_previous_autogenerated_summary=False), False)
    
    # With overwrite=True, it should return True
    test_eq(notation_note_should_be_summarized(auto_note, v, overwrite_previous_autogenerated_summary=True), True)

# 3. Test behavior when main note is missing
# (Assuming 'non_existent_main' is referenced in a notation note)
missing_main_note = VaultNote(test_vault, name='notation_with_missing_main_reference')
if missing_main_note.exists():
    test_eq(notation_note_should_be_summarized(missing_main_note, test_vault), False)

In [ ]:
#| export

def generate_notation_summary(
    model: SupportedLLM,
    excerpt_text: str,
    target_symbol: str,
    config: Optional[dict] = None,
    max_context: int = 4096,
    verbose: bool = True,
    return_thoughts: bool = False,
    system_prompt: str = NOTATION_SUMMARIZATION_SYSTEM_PROMPT,
) -> str | LLMResponse:
    
    # 1. Setup Task-Specific Overhead
    # Estimate tokens for system prompt + target formatting + safety
    reserved = 1500 # (Prompt ~800 + Target ~100 + Completion ~500 + Safety)
    
    # 2. Use Core Truncation
    truncated_text = smart_truncate(excerpt_text, model, max_context, reserved, verbose)
    
    # 3. Construct Task-Specific Message
    user_content = f"Text:\n{truncated_text}\n\nTarget Symbol: {target_symbol}\nOutput:"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]

    # 4. Delegate Execution to Core
    try:
        raw_output = call_llm(model, messages, config, verbose)
        return process_llm_response(raw_output, return_thoughts)
    except Exception as e:
        if verbose: print(f"Summary Error: {e}")
        return ""

In [ ]:
#| exec
from types import SimpleNamespace
from typing import TypedDict

class SummaryResponse(TypedDict):
    thoughts: str
    output: str

class MockLLM:
    def tokenize(self, text): 
        # Return LIST of token IDs (what real LLMs return)
        return list(range(len(text) // 4 + 1))  # Mock token IDs
    
    def respond(self, messages, config=None):
        return """<think>First identify Gal(L/K) in text. Then summarize its role in context.</think>
Gal(L/K) is the Galois group of the extension L/K, measuring symmetries of the field extension."""

mock_llm = MockLLM()
NOTATION_SUMMARIZATION_SYSTEM_PROMPT = "Summarize the mathematical role of the target symbol in the given text."

# Now works!
summary1 = generate_notation_summary(
    mock_llm, 
    "Let L/K be Galois. Gal(L/K) acts on roots.", 
    "Gal(L/K)"
)
print("Summary1:", repr(summary1))
test_eq(len(mock_llm.tokenize("test")), 2)  # len() works on list

Summary1: 'Gal(L/K) is the Galois group of the extension L/K, measuring symmetries of the field extension.'


In [ ]:
#| exec
# Example 2: Return thoughts
response2 = generate_notation_summary(
    mock_llm, 
    "The étale cohomology computes Gal(L/K).", 
    "Gal(L/K)",
    return_thoughts=True
)
test_eq(response2["thoughts"], "First identify Gal(L/K) in text. Then summarize its role in context.")
test_eq(response2["output"], "Gal(L/K) is the Galois group of the extension L/K, measuring symmetries of the field extension.")

In [ ]:
#| exec
# Example 3: Truncation
long_text = "Mathematical text about Galois groups..." * 200
summary3 = generate_notation_summary(mock_llm, long_text, "Gal(L/K)", verbose=True)
print("Truncated:", len(summary3) > 0)

Truncated: True


In [ ]:
#| hide
from fastcore.test import *

mock_llm_no_thoughts = MockLLM()
mock_llm_no_thoughts.respond = lambda s,m,c: "Direct answer"
mock_llm_no_thoughts.tokenize = lambda s,t: list(range(len(t)//4 + 1))

# Tokenization works
assert len(mock_llm.tokenize("test text")) > 0
assert isinstance(mock_llm.tokenize("test"), list)

# Function works
result = generate_notation_summary(mock_llm, "text", "symbol")
assert isinstance(result, str)
assert len(result) > 0

# Return types
str_result = generate_notation_summary(mock_llm, "text", "symbol", return_thoughts=False)
assert isinstance(str_result, str)

dict_result = generate_notation_summary(mock_llm, "text", "symbol", return_thoughts=True)
assert isinstance(dict_result, dict)
assert "thoughts" in dict_result
assert "output" in dict_result

# Error handling
class FailingLLM:
    def tokenize(self, t): return []
    def respond(self, m, c): raise ValueError("fail")
assert generate_notation_summary(FailingLLM(), "text", "symbol") == ""

Summary Error: FailingLLM.respond() got an unexpected keyword argument 'config'
